# Week 1: The Contract Before the Model: Variable Annuities and Riders

**Project:** *Valuing Variable Annuity Guarantees Under Market Crashes and Regime Shifts*  
**Mentor:** Hao Quan  
**Notebook type:** all-in-one lesson, lab, exercises, and worked solutions

> **Driving question:** What does a real variable annuity promise, and which parts will our model keep?

## Learning goals

- Explain the accumulation and payout phases of a variable annuity.
- Distinguish account value, benefit base, guarantee amount, and rider charge.
- Compare GMAB, GMDB, GMIB, GMWB, and GLWB at a conceptual level.
- Translate prospectus language into a deterministic cash-flow model.
- State clearly which real-contract features the project omits.

## How to use this notebook

1. Complete the assigned reading before the weekly meeting.
2. Read the explanation cells and predict each result before running the code.
3. Run the notebook from top to bottom. Every random experiment uses a fixed seed.
4. Attempt the exercises before opening the worked-solution section.
5. Write a 150–250 word interpretation of the main result in your own words.

This notebook is educational. It simplifies real contracts and is not an insurance
quotation, investment recommendation, or complete actuarial valuation.

## Assigned reading

- [SEC Investor Bulletin: Variable Annuities](https://www.investor.gov/introduction-investing/general-resources/news-alerts/alerts-bulletins/investor-bulletins/updated-5) — read the full bulletin; list the product phases, benefits, fees, and risks
- [Penn 2026 Updating Summary Prospectus](https://www.sec.gov/Archives/edgar/data/928880/000119312526161121/d67464d497vpu.htm) — read pp. 2–9: definitions, fees, risks, and restrictions
- [Penn Freedom Variable Annuity Prospectus](https://www.sec.gov/Archives/edgar/data/702184/000119312522115288/d269677d485bpos.htm) — read pp. 51–54: GMAB, step-up example, withdrawals, and GMWB
- [Empire Life Class Plus 3.0 Information Folder and Contract Provisions](https://www.empire.ca/docs/pdf/INV-1755-ClassPlus3-EN-web.pdf) — read pp. 1–2 and 6–15; compare Canadian GWB/income-base language with the U.S. rider terms


## 1. Read the legal object carefully

A variable annuity (VA) is a contract with an insurer. During the **accumulation
phase**, purchase payments are allocated to investment options and the contract
value changes with investment performance and charges. During the **payout
phase**, value may be converted into annuity payments. Optional **riders** amend
the base contract and generally carry an additional charge.

The Penn 2026 summary prospectus makes several distinctions that matter for our
model:

| Prospectus idea | Plain-language meaning | Project notation |
|---|---|---|
| Purchase payment | Money paid into the contract | $P$ |
| Contract/account value | Invested value after returns and charges | $A_t$ |
| Benefit base | Bookkeeping quantity used to calculate a guarantee | $B_t$ |
| Guarantee amount | Minimum amount tested on a specified date | $G_T$ |
| Rider | Optional amendment providing an added benefit | payoff rule |
| Rider charge | Charge for the optional benefit | $lpha_r$ |
| Surrender | Full exit before annuitization | omitted |
| Reset/step-up | Possible increase in a benefit base | omitted in core model |

**The most important warning:** a benefit base is usually **not cash that can be
withdrawn**. It is an accounting quantity used by a rider formula.


## 2. Rider map

- **GMAB — Guaranteed Minimum Accumulation Benefit:** tests the account at a
  specified future date and tops it up if it is below the guarantee.
- **GMDB — Guaranteed Minimum Death Benefit:** makes a death-contingent payment
  to a beneficiary if the death-benefit amount exceeds the account.
- **GMIB — Guaranteed Minimum Income Benefit:** guarantees a basis for
  annuitization or a minimum income stream, subject to contract rules.
- **GMWB — Guaranteed Minimum Withdrawal Benefit:** supports withdrawals up to
  a specified benefit amount or schedule even when investment performance is poor.
- **GLWB — Guaranteed Lifetime Withdrawal Benefit:** a lifetime form of
  withdrawal guarantee, often based on age and a benefit base.

The core project values only a **stylized return-of-premium GMAB**:

$$
L_T=(G_T-A_T)^+ = \max(G_T-A_T,0).
$$

$L_T$ is the insurer's gross maturity top-up. The customer's total value at
maturity, before other contract details, is $A_T+L_T=\max(A_T,G_T)$.


In [ ]:
import math
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
np.set_printoptions(precision=4, suppress=True)

SEED = 20260730
rng = np.random.default_rng(SEED)
print(f"Reproducible random seed: {SEED}")


## 3. A deterministic account roll-forward

Before adding Brownian motion, use a fixed sequence of annual returns. We deduct
charges continuously over each year, so one year's update is

$$A_{t+1}=A_t(1+R_{t+1})e^{-\alpha},$$

where $\alpha$ is the total annual account charge. This is a teaching
approximation; actual contracts may assess charges daily, against different
bases, and with product-specific rules.


In [ ]:
def roll_account(initial_premium, annual_returns, annual_charge):
    """Deterministic account path with returns followed by continuous fee drag."""
    if initial_premium <= 0:
        raise ValueError("initial_premium must be positive")
    if not 0 <= annual_charge < 1:
        raise ValueError("annual_charge must be between 0 and 1")

    values = [float(initial_premium)]
    for annual_return in annual_returns:
        if annual_return <= -1:
            raise ValueError("an annual return cannot be below -100%")
        values.append(values[-1] * (1 + annual_return) * np.exp(-annual_charge))
    return np.asarray(values)


annual_returns = np.array([0.08, -0.32, 0.18, 0.10, -0.06, 0.12, 0.05, 0.07, -0.04, 0.09])
premium = 100_000.0
total_charge = 0.014 + 0.0015 + 0.0090  # base + fund + illustrative rider charge
account = roll_account(premium, annual_returns, total_charge)

account_table = pd.DataFrame({
    "year": np.arange(len(account)),
    "account_value": account,
})
print(account_table.to_string(index=False, formatters={"account_value": "${:,.2f}".format}))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(account_table["year"], account_table["account_value"], marker="o", label="Account value")
ax.axhline(premium, color="black", linestyle="--", label="Return-of-premium guarantee")
ax.set(title="A deterministic VA account path", xlabel="Year", ylabel="Dollars")
ax.legend()
plt.show()


## 4. Translate three riders into simple incremental payoffs

These functions are intentionally simplified. Their purpose is to clarify the
trigger event—not to reproduce an actual product.


In [ ]:
def gmab_top_up(account_at_maturity, guarantee):
    return np.maximum(guarantee - np.asarray(account_at_maturity), 0.0)


def gmdb_increment(account_at_death, death_benefit_base):
    return np.maximum(death_benefit_base - np.asarray(account_at_death), 0.0)


def simplified_gmwb_schedule(withdrawal_base, annual_rate):
    """Level guaranteed withdrawals until the initial base is exhausted."""
    if withdrawal_base <= 0 or not 0 < annual_rate <= 1:
        raise ValueError("use a positive base and a rate in (0, 1]")
    annual_amount = withdrawal_base * annual_rate
    full_years = int(np.floor(withdrawal_base / annual_amount + 1e-12))
    payments = [annual_amount] * full_years
    residual = withdrawal_base - sum(payments)
    if residual > 1e-10:
        payments.append(residual)
    return np.asarray(payments)


terminal_account = account[-1]
gmab = float(gmab_top_up(terminal_account, premium))
gmdb = float(gmdb_increment(account[6], premium))
gmwb = simplified_gmwb_schedule(100_000, 0.05)

print(f"Terminal account: ${terminal_account:,.2f}")
print(f"Stylized GMAB top-up at year 10: ${gmab:,.2f}")
print(f"Stylized GMDB increment if death occurs in year 6: ${gmdb:,.2f}")
print(f"Simplified 5% GMWB: {len(gmwb)} payments of ${gmwb[0]:,.2f}")

assert gmab >= 0
assert gmab_top_up(120_000, 100_000) == 0
assert gmab_top_up(80_000, 100_000) == 20_000


## 5. Reproduce a real prospectus calculation

The Penn Freedom prospectus explains that a withdrawal can reduce the GMAB base
**proportionally**, not dollar for dollar:

$$
\text{base reduction}
=B_{t^-}\frac{w}{A_{t^-}}.
$$

The published example uses a $7,500 withdrawal, a $100,000 benefit base, and
account values of $110,000 and $90,000. Reproducing the arithmetic is a useful
contract-reading check.


In [ ]:
def proportional_base_reduction(benefit_base, account_before, withdrawal):
    if benefit_base < 0 or account_before <= 0:
        raise ValueError("invalid base or account value")
    if not 0 <= withdrawal <= account_before:
        raise ValueError("withdrawal must be between 0 and the account value")
    reduction = benefit_base * withdrawal / account_before
    return reduction, benefit_base - reduction


rows = []
for account_before in (110_000, 90_000):
    reduction, new_base = proportional_base_reduction(100_000, account_before, 7_500)
    rows.append({
        "account_before": account_before,
        "withdrawal": 7_500,
        "base_reduction": reduction,
        "new_base": new_base,
    })

prospectus_check = pd.DataFrame(rows)
print(prospectus_check.to_string(index=False, formatters={
    c: "${:,.2f}".format for c in prospectus_check.columns
}))

assert np.isclose(rows[0]["base_reduction"], 6_818.18181818)
assert np.isclose(rows[1]["base_reduction"], 8_333.33333333)


## 6. Scenario laboratory: the same guarantee under different markets

The guarantee is valuable only when the terminal account falls below $G_T$.
Notice that a deeper loss increases the **incremental** insurer payment.


In [ ]:
scenarios = pd.DataFrame({
    "scenario": ["strong bull", "modest growth", "flat after fees", "moderate loss", "deep crash"],
    "terminal_account": [160_000, 118_000, 100_000, 82_000, 45_000],
})
scenarios["gmab_top_up"] = gmab_top_up(scenarios["terminal_account"], premium)
scenarios["customer_total_at_maturity"] = (
    scenarios["terminal_account"] + scenarios["gmab_top_up"]
)
print(scenarios.to_string(index=False, formatters={
    "terminal_account": "${:,.0f}".format,
    "gmab_top_up": "${:,.0f}".format,
    "customer_total_at_maturity": "${:,.0f}".format,
}))

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(scenarios))
ax.bar(x, scenarios["terminal_account"], label="Account")
ax.bar(x, scenarios["gmab_top_up"], bottom=scenarios["terminal_account"], label="GMAB top-up")
ax.axhline(premium, color="black", linestyle="--", linewidth=1)
ax.set_xticks(x, scenarios["scenario"], rotation=20)
ax.set(ylabel="Dollars", title="Account value and incremental GMAB payment")
ax.legend()
plt.tight_layout()
plt.show()

assert (scenarios["gmab_top_up"] >= 0).all()
assert np.all(np.diff(gmab_top_up(np.array([40_000, 60_000, 80_000]), premium)) <= 0)


## 7. Prospectus scavenger hunt

Before reading the answer key, locate each item in the assigned prospectuses.

1. What is the difference between a Variable Investment Option and the Separate Account?
2. On what basis can optional-benefit charges be assessed?
3. Why can a surrender reduce more than just the account value?
4. What events can terminate a GMAB rider?
5. Why is a rider guarantee not equivalent to a bank deposit guarantee?

### Answer key

1. A Variable Investment Option is a subaccount/investment choice; the Separate
   Account is the segregated insurer account divided into such subaccounts.
2. Depending on the rider, the charge may be based on account value or a
   benefit base; the actual contract controls.
3. Surrender charges, taxes, penalties, and reductions or termination of
   optional benefits can all matter.
4. Product-specific examples include the end of the benefit period, surrender,
   annuitization, death, or a requested termination.
5. Contract guarantees depend on the insurer's claims-paying ability and are
   not FDIC insurance.


## 8. Exercises — attempt before the worked solutions

1. Remove the illustrative rider charge from the deterministic path. Calculate
   the difference in terminal account value.
2. For $B=125{,}000$, $A=95{,}000$, and $w=10{,}000$, calculate the
   proportional benefit-base reduction.
3. Create a three-row table that distinguishes the trigger, timing, and
   beneficiary of GMAB, GMDB, and GMWB.
4. Write a one-sentence mathematical definition of the stylized GMAB top-up.


## 9. Worked solutions


In [ ]:
# Exercise 1
account_without_rider_charge = roll_account(
    premium, annual_returns, total_charge - 0.0090
)
fee_difference = account_without_rider_charge[-1] - account[-1]
print(f"Terminal-value difference without the 0.90% rider charge: ${fee_difference:,.2f}")

# Exercise 2
reduction, revised_base = proportional_base_reduction(125_000, 95_000, 10_000)
print(f"Base reduction: ${reduction:,.2f}; revised base: ${revised_base:,.2f}")

# Exercise 3
rider_comparison = pd.DataFrame([
    ["GMAB", "Account below guarantee", "Specified accumulation date", "Contract owner"],
    ["GMDB", "Covered death and account below death benefit", "At death claim", "Beneficiary"],
    ["GMWB", "Eligible scheduled withdrawals", "During withdrawal phase", "Contract owner"],
], columns=["rider", "simplified_trigger", "timing", "recipient"])
print(rider_comparison.to_string(index=False))

# Exercise 4: L_T = max(G_T - A_T, 0).
assert np.isclose(reduction, 13_157.89473684)


## 10. Scope statement for the rest of the project

From Week 4 onward, the core contract uses $A_0=P=100$, $T=10$, and
$G_T=P$. It has one fund, continuous account charges, no extra premiums,
withdrawals, mortality, lapses, surrender, taxes, resets, ratchets, or
annuitization choices.

Real prospectuses motivate the mechanics, but **we are not valuing the Penn
contract**. We are valuing a stylized gross maturity shortfall so that model
risk can be studied cleanly.

### Weekly submission

- Completed notebook
- One-page contract-to-model memo
- Ten-term glossary
- A paragraph explaining why account value and benefit base are not the same
